In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn
import torch

In [35]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [36]:
train_df.shape

(42000, 785)

In [37]:
X = train_df.drop("label", axis=1).values
y = train_df.values

In [38]:
X = X/255
test_df = test_df.values/255

In [39]:
X = X.reshape(-1, 1, 28, 28)
test_df = test_df.reshape(-1, 1, 28, 28)

In [40]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.from_numpy(X_train).float()
X_val = torch.from_numpy(X_val).float()
y_train = torch.from_numpy(y_train).float().squeeze()
y_val = torch.from_numpy(y_val).float().squeeze()

In [41]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.Linear(128, 10),
        )

    def forward(self, X):
        return self.model(X)




In [ ]:
model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

: 

In [ ]:
epochs = 1

for epoch in range(epochs):
    model.train()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        preds = torch.argmax(val_logits, dim=1)
        acc = (preds == y_val).float().mean()

    print(f"Epoch {epoch} | Loss {loss:.4f} | Val Acc {acc:.4f}")

In [ ]:
test_tensor = torch.tensor(test_df, dtype=torch.float32)
with torch.no_grad():
    logits = model(test_tensor)
    preds = torch.argmax(logits, dim=1)
submission = pd.DataFrame({
    "ImageId": range(1,len(preds)+1),
    "Label": preds
})